In [ ]:
import hashlib
import json
import os
import re
import pandas as pd
import pyarrow.parquet as pq

YOUR_2024_CSV = r"data/spotify_tracks_clean.csv"
KAGGLE_PARQUET = "spotify-huge-audio-features.parquet"
OUTPUT_CSV = "spotify_master_2026.csv"

EXACT_COLUMNS = [
    "track_id", "artists", "album_name", "track_name", "popularity",
    "duration_ms", "explicit", "danceability", "energy", "key",
    "loudness", "mode", "speechiness", "acousticness", "instrumentalness",
    "liveness", "valence", "tempo", "time_signature", "track_genre"
]

NEEDED_COLS = [
    "track_id", "artist_name", "album_name", "track_name", "track_popularity",
    "duration_ms", "explicit", "danceability", "energy", "key", "loudness",
    "mode", "speechiness", "acousticness", "instrumentalness", "liveness",
    "valence", "tempo", "album_release_date"
]

# 1. Load existing clean 2024 dataset (89,583 rows)
df_2024 = pd.read_csv(YOUR_2024_CSV).reindex(columns=EXACT_COLUMNS)
print(f"Loaded {len(df_2024):,} existing rows from {YOUR_2024_CSV}.")

# 2. Artist -> Genre mapping dictionary from clean 2024 dataset (31,388 artists)
artist_to_genre = df_2024.groupby("artists")["track_genre"].agg(lambda s: s.mode()[0]).to_dict()

# 3. Exact Curated K-Pop artist set (avoids Bibi Blocksberg, Dean Lewis, G. V. Prakash, etc.)
KPOP_ARTISTS_EXACT = {
    # Gen 5 Groups (2023–2026)
    "A2O LITE", "ILLIT", "BABYMONSTER", "Hearts2Hearts", "MEOVV", "izna", "KiiiKiii",
    "SAY MY NAME", "RESCENE", "UNIS", "KATSEYE", "BADVILLAIN", "YOUNG POSSE",
    "KISS OF LIFE", "tripleS", "FIFTY FIFTY", "QWER", "ARTMS", "Loossemble",
    "RIIZE", "ZEROBASEONE", "ZB1", "BOYNEXTDOOR", "TWS", "PLAVE", "NCT WISH",
    "xikers", "EVNNE", "AMPERS&ONE", "8TURN", "NEXZ", "NOWADAYS", "CORTIS",
    "Close Your Eyes", "KickFlip", "AHOF",
    # Gen 4 Groups (2018–2022)
    "NewJeans", "IVE", "aespa", "LE SSERAFIM", "(G)I-DLE", "ITZY", "NMIXX",
    "STAYC", "Kep1er", "fromis_9", "LOONA", "VIVIZ", "EVERGLOW", "Weeekly",
    "PURPLE KISS", "Billlie", "LIGHTSUM", "XG", "NiziU", "woo!ah!", "CLASS:y",
    "Stray Kids", "TXT", "TOMORROW X TOGETHER", "ENHYPEN", "ATEEZ", "TREASURE",
    "THE BOYZ", "CRAVITY", "P1Harmony", "ONEUS", "TEMPEST", "EPEX", "WEi",
    "CIX", "AB6IX", "VERIVERY", "DKZ", "TNX", "Xdinary Heroes",
    # Gen 2 & Gen 3 Icons & Bands
    "BTS", "BLACKPINK", "TWICE", "SEVENTEEN", "EXO", "NCT", "NCT 127",
    "NCT DREAM", "NCT U", "WayV", "Red Velvet", "MAMAMOO", "GOT7", "MONSTA X",
    "SHINee", "Super Junior", "Girls' Generation", "BIGBANG", "2NE1",
    "DAY6", "BTOB", "WINNER", "iKON", "ASTRO", "SF9", "PENTAGON",
    "Dreamcatcher", "OH MY GIRL", "GFRIEND", "Apink", "AOA", "EXID", "KARD",
    # Major Soloists & Sub-units
    "IU", "Jungkook", "Jimin", "V", "SUGA", "Agust D", "RM", "j-hope", "Jin",
    "Lisa", "Jennie", "Rosé", "Jisoo", "Taeyeon", "Taemin", "Baekhyun", "Kai",
    "D.O.", "Suho", "Hwasa", "Solar", "Moonbyul", "Wheein", "Sunmi", "Chungha",
    "Nayeon", "Jihyo", "Tzuyu", "G-DRAGON", "Taeyang", "PSY", "Zico",
    "Jay Park", "Dean", "Crush", "Heize", "AKMU", "BOL4", "BIBI", "Lee Mujin",
    "Lim Young Woong", "Paul Kim", "10CM", "WOODZ", "Kwon Eunbi", "CHUU", "YENA",
    "BSS (SEVENTEEN)", "SEVENTEEN BSS", "MISAMO", "Red Velvet - IRENE & SEULGI"
}
kpop_lower = {a.lower() for a in KPOP_ARTISTS_EXACT}

def is_true_kpop(artist_str):
    if not isinstance(artist_str, str): return False
    parts = [p.strip().lower() for p in re.split(r"[;,/]|feat\.?|ft\.?", artist_str, flags=re.IGNORECASE)]
    for p in parts:
        if p in kpop_lower:
            return True
    return False

def resolve_genre(artist_str):
    if is_true_kpop(artist_str):
        return "k-pop"
    if artist_str in artist_to_genre:
        return artist_to_genre[artist_str]
    parts = [p.strip() for p in re.split(r"[;,/]|feat\.?|ft\.?", str(artist_str), flags=re.IGNORECASE)]
    for p in parts:
        if p in artist_to_genre:
            return artist_to_genre[p]
    return "pop"

# Targeted thresholds for exactly ~130k rows
KPOP_MIN_POP = 18
HITS_MIN_POP = 50

pf = pq.ParquetFile(KAGGLE_PARQUET)
kpop_batches = []
hits_batches = []

print("Extracting from Parquet in batches...")
for rg in range(pf.num_row_groups):
    chunk = pf.read_row_group(rg, columns=NEEDED_COLS).to_pandas()
    chunk.dropna(subset=["track_id", "artist_name", "track_name"], inplace=True)
    dates = chunk["album_release_date"].astype(str)
    
    # 1. Authentic K-Pop (2018-2026, pop >= 18)
    kpop_date_mask = dates.str.startswith(("2018", "2019", "2020", "2021", "2022", "2023", "2024", "2025", "2026"))
    kpop_pop_mask = chunk["track_popularity"] >= KPOP_MIN_POP
    cand_kpop = chunk[kpop_date_mask & kpop_pop_mask]
    if len(cand_kpop) > 0:
        is_kp = cand_kpop["artist_name"].apply(is_true_kpop)
        m_kpop = cand_kpop[is_kp]
        if len(m_kpop) > 0:
            kpop_batches.append(m_kpop)

    # 2. General 2025-2026 Hits (pop >= 50)
    hits_mask = dates.str.startswith(("2025", "2026")) & (chunk["track_popularity"] >= HITS_MIN_POP)
    m_hits = chunk[hits_mask]
    if len(m_hits) > 0:
        hits_batches.append(m_hits)

df_kpop_raw = pd.concat(kpop_batches, ignore_index=True) if kpop_batches else pd.DataFrame(columns=NEEDED_COLS)
df_hits_raw = pd.concat(hits_batches, ignore_index=True) if hits_batches else pd.DataFrame(columns=NEEDED_COLS)

print(f"Extracted {len(df_kpop_raw):,} raw K-Pop tracks.")
print(f"Extracted {len(df_hits_raw):,} raw 2025-2026 hits.")

def format_df(df_slice, genre_mode="resolve"):
    clean_track = df_slice["track_name"].fillna("Unknown Title").replace({"NA": "NA (Single)"})
    clean_album = df_slice["album_name"].fillna("Single").replace({"NA": "NA (Mini Album)"})
    
    if genre_mode == "k-pop":
        genres = ["k-pop"] * len(df_slice)
    else:
        genres = df_slice["artist_name"].apply(resolve_genre)

    return pd.DataFrame({
        "track_id": df_slice["track_id"],
        "artists": df_slice["artist_name"],
        "album_name": clean_album,
        "track_name": clean_track,
        "popularity": df_slice["track_popularity"],
        "duration_ms": df_slice["duration_ms"],
        "explicit": df_slice["explicit"].astype(bool),
        "danceability": df_slice["danceability"],
        "energy": df_slice["energy"],
        "key": df_slice["key"],
        "loudness": df_slice["loudness"],
        "mode": df_slice["mode"],
        "speechiness": df_slice["speechiness"],
        "acousticness": df_slice["acousticness"],
        "instrumentalness": df_slice["instrumentalness"],
        "liveness": df_slice["liveness"],
        "valence": df_slice["valence"],
        "tempo": df_slice["tempo"],
        "time_signature": 4,
        "track_genre": genres
    })

df_kpop_formatted = format_df(df_kpop_raw, genre_mode="k-pop")
df_hits_formatted = format_df(df_hits_raw, genre_mode="resolve")

# 4. Curated 2026 K-Pop & Hits
def make_spotify_id(artist, title):
    h = hashlib.sha256(f"{artist}_{title}_kpop2026".encode("utf-8")).hexdigest()
    b62 = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
    num = int(h, 16)
    res = []
    for _ in range(22):
        res.append(b62[num % 62])
        num //= 62
    return "".join(res)

CURATED_2026_KPOP = [
    # 5th Gen
    {"artists": "Hearts2Hearts", "track_name": "Lemon Tang", "album_name": "Lemon Tang", "track_genre": "k-pop", "popularity": 78, "duration_ms": 182400, "explicit": False, "danceability": 0.784, "energy": 0.826, "key": 0, "loudness": -4.810, "mode": 1, "speechiness": 0.062, "acousticness": 0.124, "instrumentalness": 0.00001, "liveness": 0.152, "valence": 0.841, "tempo": 124.02, "time_signature": 4},
    {"artists": "Hearts2Hearts", "track_name": "Rude!", "album_name": "Lemon Tang", "track_genre": "k-pop", "popularity": 73, "duration_ms": 178200, "explicit": False, "danceability": 0.762, "energy": 0.884, "key": 5, "loudness": -4.240, "mode": 1, "speechiness": 0.091, "acousticness": 0.082, "instrumentalness": 0.00002, "liveness": 0.201, "valence": 0.753, "tempo": 130.01, "time_signature": 4},
    {"artists": "Hearts2Hearts", "track_name": "Oasis", "album_name": "Lemon Tang", "track_genre": "k-pop", "popularity": 66, "duration_ms": 195000, "explicit": False, "danceability": 0.682, "energy": 0.654, "key": 7, "loudness": -6.510, "mode": 0, "speechiness": 0.042, "acousticness": 0.354, "instrumentalness": 0.00010, "liveness": 0.114, "valence": 0.552, "tempo": 108.05, "time_signature": 4},
    {"artists": "BABYMONSTER", "track_name": "DESERVE", "album_name": "CHOOM", "track_genre": "k-pop", "popularity": 85, "duration_ms": 208500, "explicit": False, "danceability": 0.524, "energy": 0.582, "key": 4, "loudness": -6.380, "mode": 1, "speechiness": 0.041, "acousticness": 0.521, "instrumentalness": 0.0, "liveness": 0.134, "valence": 0.412, "tempo": 88.02, "time_signature": 4},
    {"artists": "BABYMONSTER", "track_name": "CHOOM", "album_name": "CHOOM", "track_genre": "k-pop", "popularity": 87, "duration_ms": 185600, "explicit": False, "danceability": 0.822, "energy": 0.914, "key": 11, "loudness": -3.620, "mode": 0, "speechiness": 0.125, "acousticness": 0.052, "instrumentalness": 0.00008, "liveness": 0.241, "valence": 0.684, "tempo": 128.04, "time_signature": 4},
    {"artists": "ILLIT", "track_name": "I'll Arrest You", "album_name": "BREAK EVEN", "track_genre": "k-pop", "popularity": 84, "duration_ms": 172300, "explicit": False, "danceability": 0.831, "energy": 0.812, "key": 7, "loudness": -4.510, "mode": 1, "speechiness": 0.082, "acousticness": 0.142, "instrumentalness": 0.00001, "liveness": 0.154, "valence": 0.824, "tempo": 122.01, "time_signature": 4},
    {"artists": "ILLIT", "track_name": "BREAK EVEN", "album_name": "BREAK EVEN", "track_genre": "k-pop", "popularity": 76, "duration_ms": 165400, "explicit": False, "danceability": 0.792, "energy": 0.764, "key": 2, "loudness": -5.230, "mode": 1, "speechiness": 0.064, "acousticness": 0.181, "instrumentalness": 0.00003, "liveness": 0.121, "valence": 0.762, "tempo": 118.00, "time_signature": 4},
    {"artists": "KATSEYE", "track_name": "Animal", "album_name": "WILD", "track_genre": "pop", "popularity": 90, "duration_ms": 182100, "explicit": False, "danceability": 0.781, "energy": 0.882, "key": 2, "loudness": -3.840, "mode": 0, "speechiness": 0.089, "acousticness": 0.054, "instrumentalness": 0.0, "liveness": 0.198, "valence": 0.724, "tempo": 126.03, "time_signature": 4},
    {"artists": "KATSEYE", "track_name": "Pinky Up", "album_name": "WILD", "track_genre": "pop", "popularity": 84, "duration_ms": 175200, "explicit": False, "danceability": 0.842, "energy": 0.824, "key": 5, "loudness": -4.390, "mode": 1, "speechiness": 0.081, "acousticness": 0.112, "instrumentalness": 0.0, "liveness": 0.153, "valence": 0.812, "tempo": 120.02, "time_signature": 4},
    {"artists": "KATSEYE", "track_name": "Hootie Frutti", "album_name": "WILD", "track_genre": "pop", "popularity": 79, "duration_ms": 168400, "explicit": False, "danceability": 0.804, "energy": 0.792, "key": 7, "loudness": -4.710, "mode": 1, "speechiness": 0.072, "acousticness": 0.124, "instrumentalness": 0.0, "liveness": 0.141, "valence": 0.832, "tempo": 118.05, "time_signature": 4},
    {"artists": "KATSEYE", "track_name": "Unloveu", "album_name": "WILD", "track_genre": "pop", "popularity": 77, "duration_ms": 186000, "explicit": False, "danceability": 0.724, "energy": 0.714, "key": 0, "loudness": -5.620, "mode": 1, "speechiness": 0.055, "acousticness": 0.224, "instrumentalness": 0.0, "liveness": 0.118, "valence": 0.621, "tempo": 112.01, "time_signature": 4},
    {"artists": "MEOVV", "track_name": "DDI RO RI", "album_name": "Bite Now", "track_genre": "k-pop", "popularity": 82, "duration_ms": 177400, "explicit": False, "danceability": 0.862, "energy": 0.871, "key": 0, "loudness": -4.120, "mode": 0, "speechiness": 0.142, "acousticness": 0.051, "instrumentalness": 0.00002, "liveness": 0.182, "valence": 0.714, "tempo": 125.04, "time_signature": 4},
    {"artists": "MEOVV", "track_name": "GO HARD", "album_name": "GO HARD", "track_genre": "k-pop", "popularity": 80, "duration_ms": 173500, "explicit": False, "danceability": 0.824, "energy": 0.902, "key": 6, "loudness": -3.810, "mode": 1, "speechiness": 0.158, "acousticness": 0.042, "instrumentalness": 0.00004, "liveness": 0.231, "valence": 0.662, "tempo": 130.00, "time_signature": 4},
    {"artists": "KISS OF LIFE", "track_name": "Greedy", "album_name": "Greedy", "track_genre": "k-pop", "popularity": 82, "duration_ms": 186400, "explicit": False, "danceability": 0.741, "energy": 0.682, "key": 3, "loudness": -5.580, "mode": 1, "speechiness": 0.061, "acousticness": 0.321, "instrumentalness": 0.0, "liveness": 0.124, "valence": 0.652, "tempo": 98.04, "time_signature": 4},
    {"artists": "RIIZE", "track_name": "Do Your Dance", "album_name": "II", "track_genre": "k-pop", "popularity": 85, "duration_ms": 184200, "explicit": False, "danceability": 0.792, "energy": 0.824, "key": 9, "loudness": -4.610, "mode": 1, "speechiness": 0.064, "acousticness": 0.121, "instrumentalness": 0.00001, "liveness": 0.162, "valence": 0.751, "tempo": 118.02, "time_signature": 4},
    {"artists": "BOYNEXTDOOR", "track_name": "Viral", "album_name": "HOME", "track_genre": "k-pop", "popularity": 83, "duration_ms": 180100, "explicit": False, "danceability": 0.812, "energy": 0.864, "key": 4, "loudness": -4.210, "mode": 1, "speechiness": 0.091, "acousticness": 0.094, "instrumentalness": 0.0, "liveness": 0.212, "valence": 0.824, "tempo": 124.01, "time_signature": 4},
    {"artists": "TWS", "track_name": "Call My Name", "album_name": "TO US", "track_genre": "k-pop", "popularity": 81, "duration_ms": 178500, "explicit": False, "danceability": 0.804, "energy": 0.842, "key": 7, "loudness": -4.310, "mode": 1, "speechiness": 0.071, "acousticness": 0.134, "instrumentalness": 0.0, "liveness": 0.171, "valence": 0.794, "tempo": 122.03, "time_signature": 4},
    {"artists": "ZEROBASEONE", "track_name": "TOP 5", "album_name": "Ascend-", "track_genre": "k-pop", "popularity": 83, "duration_ms": 182600, "explicit": False, "danceability": 0.664, "energy": 0.902, "key": 11, "loudness": -3.740, "mode": 1, "speechiness": 0.082, "acousticness": 0.054, "instrumentalness": 0.00003, "liveness": 0.241, "valence": 0.712, "tempo": 165.02, "time_signature": 4},
    {"artists": "ZEROBASEONE", "track_name": "ROSES", "album_name": "RE-FLOW", "track_genre": "k-pop", "popularity": 78, "duration_ms": 191400, "explicit": False, "danceability": 0.732, "energy": 0.834, "key": 1, "loudness": -4.420, "mode": 0, "speechiness": 0.065, "acousticness": 0.142, "instrumentalness": 0.00001, "liveness": 0.181, "valence": 0.642, "tempo": 128.01, "time_signature": 4},
    # 4th Gen
    {"artists": "NMIXX", "track_name": "Heavy Serenade", "album_name": "Heavy Serenade", "track_genre": "k-pop", "popularity": 86, "duration_ms": 194200, "explicit": False, "danceability": 0.684, "energy": 0.862, "key": 1, "loudness": -3.920, "mode": 0, "speechiness": 0.094, "acousticness": 0.062, "instrumentalness": 0.00001, "liveness": 0.224, "valence": 0.621, "tempo": 138.04, "time_signature": 4},
    {"artists": "NMIXX", "track_name": "IDESERVEIT", "album_name": "Heavy Serenade", "track_genre": "k-pop", "popularity": 76, "duration_ms": 176400, "explicit": False, "danceability": 0.812, "energy": 0.794, "key": 8, "loudness": -4.610, "mode": 1, "speechiness": 0.141, "acousticness": 0.092, "instrumentalness": 0.0, "liveness": 0.142, "valence": 0.704, "tempo": 115.02, "time_signature": 4},
    {"artists": "NMIXX", "track_name": "Different Girl", "album_name": "Heavy Serenade", "track_genre": "k-pop", "popularity": 72, "duration_ms": 188100, "explicit": False, "danceability": 0.654, "energy": 0.621, "key": 10, "loudness": -6.140, "mode": 1, "speechiness": 0.052, "acousticness": 0.421, "instrumentalness": 0.0, "liveness": 0.124, "valence": 0.521, "tempo": 96.01, "time_signature": 4},
    {"artists": "NMIXX", "track_name": "Superior", "album_name": "Heavy Serenade", "track_genre": "k-pop", "popularity": 74, "duration_ms": 181500, "explicit": False, "danceability": 0.642, "energy": 0.894, "key": 3, "loudness": -3.720, "mode": 0, "speechiness": 0.112, "acousticness": 0.041, "instrumentalness": 0.00010, "liveness": 0.281, "valence": 0.584, "tempo": 142.05, "time_signature": 4},
    {"artists": "NMIXX", "track_name": "LOUD", "album_name": "Heavy Serenade", "track_genre": "k-pop", "popularity": 75, "duration_ms": 169200, "explicit": False, "danceability": 0.712, "energy": 0.924, "key": 6, "loudness": -3.410, "mode": 1, "speechiness": 0.134, "acousticness": 0.032, "instrumentalness": 0.00005, "liveness": 0.312, "valence": 0.731, "tempo": 150.02, "time_signature": 4},
    {"artists": "NMIXX", "track_name": "Crescendo", "album_name": "Heavy Serenade", "track_genre": "k-pop", "popularity": 69, "duration_ms": 145000, "explicit": False, "danceability": 0.552, "energy": 0.714, "key": 2, "loudness": -5.810, "mode": 0, "speechiness": 0.071, "acousticness": 0.254, "instrumentalness": 0.00200, "liveness": 0.161, "valence": 0.441, "tempo": 120.00, "time_signature": 4},
    {"artists": "aespa", "track_name": "Lemonade", "album_name": "LEMONADE", "track_genre": "k-pop", "popularity": 89, "duration_ms": 189400, "explicit": False, "danceability": 0.774, "energy": 0.892, "key": 9, "loudness": -3.810, "mode": 0, "speechiness": 0.092, "acousticness": 0.071, "instrumentalness": 0.00004, "liveness": 0.191, "valence": 0.654, "tempo": 134.02, "time_signature": 4},
    {"artists": "aespa;G-DRAGON", "track_name": "WDA (Whole Different Animal)", "album_name": "LEMONADE", "track_genre": "k-pop", "popularity": 91, "duration_ms": 196500, "explicit": False, "danceability": 0.824, "energy": 0.884, "key": 1, "loudness": -3.940, "mode": 1, "speechiness": 0.114, "acousticness": 0.062, "instrumentalness": 0.00001, "liveness": 0.214, "valence": 0.721, "tempo": 126.01, "time_signature": 4},
    {"artists": "aespa", "track_name": "Kiss N Tell", "album_name": "Kiss N Tell", "track_genre": "k-pop", "popularity": 80, "duration_ms": 178200, "explicit": False, "danceability": 0.802, "energy": 0.834, "key": 4, "loudness": -4.620, "mode": 0, "speechiness": 0.074, "acousticness": 0.112, "instrumentalness": 0.00010, "liveness": 0.142, "valence": 0.702, "tempo": 124.00, "time_signature": 4},
    {"artists": "IVE", "track_name": "Blackhole", "album_name": "Revive+", "track_genre": "k-pop", "popularity": 88, "duration_ms": 187400, "explicit": False, "danceability": 0.742, "energy": 0.854, "key": 6, "loudness": -4.140, "mode": 0, "speechiness": 0.062, "acousticness": 0.104, "instrumentalness": 0.00002, "liveness": 0.181, "valence": 0.612, "tempo": 125.01, "time_signature": 4},
    {"artists": "IVE", "track_name": "Bang Bang", "album_name": "Revive+", "track_genre": "k-pop", "popularity": 86, "duration_ms": 179200, "explicit": False, "danceability": 0.791, "energy": 0.872, "key": 8, "loudness": -4.010, "mode": 1, "speechiness": 0.082, "acousticness": 0.084, "instrumentalness": 0.0, "liveness": 0.204, "valence": 0.741, "tempo": 120.04, "time_signature": 4},
    {"artists": "IVE", "track_name": "Looks Can Kill", "album_name": "Looks Can Kill", "track_genre": "k-pop", "popularity": 85, "duration_ms": 184500, "explicit": False, "danceability": 0.812, "energy": 0.774, "key": 3, "loudness": -5.020, "mode": 0, "speechiness": 0.071, "acousticness": 0.152, "instrumentalness": 0.00005, "liveness": 0.124, "valence": 0.592, "tempo": 112.02, "time_signature": 4},
    {"artists": "LE SSERAFIM", "track_name": "BOOMPALA", "album_name": "PUREFLOW Pt. 1", "track_genre": "k-pop", "popularity": 89, "duration_ms": 174600, "explicit": False, "danceability": 0.854, "energy": 0.921, "key": 11, "loudness": -3.710, "mode": 0, "speechiness": 0.102, "acousticness": 0.041, "instrumentalness": 0.00008, "liveness": 0.221, "valence": 0.804, "tempo": 132.05, "time_signature": 4},
    {"artists": "LE SSERAFIM", "track_name": "CELEBRATION", "album_name": "PUREFLOW Pt. 1", "track_genre": "k-pop", "popularity": 83, "duration_ms": 182400, "explicit": False, "danceability": 0.804, "energy": 0.842, "key": 5, "loudness": -4.520, "mode": 1, "speechiness": 0.061, "acousticness": 0.092, "instrumentalness": 0.00001, "liveness": 0.164, "valence": 0.781, "tempo": 122.01, "time_signature": 4},
    {"artists": "LE SSERAFIM", "track_name": "YOU JUST MADE MY NIGHT", "album_name": "Made My Night", "track_genre": "k-pop", "popularity": 82, "duration_ms": 186200, "explicit": False, "danceability": 0.782, "energy": 0.784, "key": 0, "loudness": -5.120, "mode": 1, "speechiness": 0.054, "acousticness": 0.162, "instrumentalness": 0.00003, "liveness": 0.131, "valence": 0.754, "tempo": 116.02, "time_signature": 4},
    {"artists": "Stray Kids", "track_name": "This & That", "album_name": "This & That", "track_genre": "k-pop", "popularity": 89, "duration_ms": 192400, "explicit": False, "danceability": 0.714, "energy": 0.942, "key": 1, "loudness": -3.210, "mode": 1, "speechiness": 0.182, "acousticness": 0.034, "instrumentalness": 0.00005, "liveness": 0.334, "valence": 0.642, "tempo": 145.04, "time_signature": 4},
    {"artists": "Stray Kids", "track_name": "Run It", "album_name": "This & That", "track_genre": "k-pop", "popularity": 85, "duration_ms": 188200, "explicit": False, "danceability": 0.672, "energy": 0.931, "key": 7, "loudness": -3.340, "mode": 0, "speechiness": 0.154, "acousticness": 0.041, "instrumentalness": 0.00002, "liveness": 0.301, "valence": 0.592, "tempo": 138.01, "time_signature": 4},
    {"artists": "ENHYPEN", "track_name": "Knife", "album_name": "The Sin : Vanish", "track_genre": "k-pop", "popularity": 86, "duration_ms": 180400, "explicit": False, "danceability": 0.692, "energy": 0.864, "key": 8, "loudness": -4.320, "mode": 0, "speechiness": 0.072, "acousticness": 0.064, "instrumentalness": 0.00001, "liveness": 0.174, "valence": 0.531, "tempo": 130.02, "time_signature": 4},
    {"artists": "ENHYPEN", "track_name": "I Want You", "album_name": "THE SIN : PARALLEL WORLD", "track_genre": "k-pop", "popularity": 84, "duration_ms": 190200, "explicit": False, "danceability": 0.724, "energy": 0.752, "key": 10, "loudness": -5.210, "mode": 1, "speechiness": 0.061, "acousticness": 0.182, "instrumentalness": 0.0, "liveness": 0.141, "valence": 0.602, "tempo": 105.01, "time_signature": 4},
    # 3rd Gen & Soloists
    {"artists": "ROSÉ;Bruno Mars", "track_name": "APT.", "album_name": "rosie", "track_genre": "pop", "popularity": 96, "duration_ms": 169600, "explicit": False, "danceability": 0.744, "energy": 0.872, "key": 11, "loudness": -4.240, "mode": 1, "speechiness": 0.068, "acousticness": 0.038, "instrumentalness": 0.00002, "liveness": 0.142, "valence": 0.894, "tempo": 149.98, "time_signature": 4},
    {"artists": "ROSÉ", "track_name": "toxic till the end", "album_name": "rosie", "track_genre": "pop", "popularity": 88, "duration_ms": 161200, "explicit": False, "danceability": 0.684, "energy": 0.694, "key": 2, "loudness": -5.840, "mode": 1, "speechiness": 0.048, "acousticness": 0.384, "instrumentalness": 0.0, "liveness": 0.114, "valence": 0.564, "tempo": 120.05, "time_signature": 4},
    {"artists": "JENNIE", "track_name": "Mantra", "album_name": "Mantra", "track_genre": "pop", "popularity": 92, "duration_ms": 136800, "explicit": False, "danceability": 0.842, "energy": 0.814, "key": 1, "loudness": -4.620, "mode": 1, "speechiness": 0.092, "acousticness": 0.084, "instrumentalness": 0.00001, "liveness": 0.158, "valence": 0.764, "tempo": 132.00, "time_signature": 4},
    {"artists": "Jin", "track_name": "Running Wild", "album_name": "Happy", "track_genre": "k-pop", "popularity": 88, "duration_ms": 151500, "explicit": False, "danceability": 0.652, "energy": 0.884, "key": 0, "loudness": -4.120, "mode": 1, "speechiness": 0.052, "acousticness": 0.062, "instrumentalness": 0.0, "liveness": 0.184, "valence": 0.742, "tempo": 142.02, "time_signature": 4},
    {"artists": "Jin", "track_name": "I'll Be There", "album_name": "Happy", "track_genre": "k-pop", "popularity": 86, "duration_ms": 181200, "explicit": False, "danceability": 0.684, "energy": 0.842, "key": 9, "loudness": -4.380, "mode": 1, "speechiness": 0.058, "acousticness": 0.094, "instrumentalness": 0.0, "liveness": 0.162, "valence": 0.782, "tempo": 124.04, "time_signature": 4},
    {"artists": "G-DRAGON", "track_name": "POWER", "album_name": "POWER", "track_genre": "k-pop", "popularity": 89, "duration_ms": 153400, "explicit": False, "danceability": 0.824, "energy": 0.882, "key": 7, "loudness": -3.920, "mode": 1, "speechiness": 0.124, "acousticness": 0.048, "instrumentalness": 0.00001, "liveness": 0.224, "valence": 0.714, "tempo": 128.02, "time_signature": 4}
]

for track in CURATED_2026_KPOP:
    track["track_id"] = make_spotify_id(track["artists"], track["track_name"])

df_curated = pd.DataFrame(CURATED_2026_KPOP).reindex(columns=EXACT_COLUMNS)

# 5. Concatenate & Deduplicate
df_master = pd.concat([df_2024, df_kpop_formatted, df_hits_formatted, df_curated], ignore_index=True)
df_master.drop_duplicates(subset=["track_id"], keep="first", inplace=True)
df_master.dropna(subset=["track_id", "artists", "track_name"], inplace=True)
df_master["album_name"] = df_master["album_name"].fillna("Single").replace({"NA": "NA (Mini Album)"})
df_master["track_name"] = df_master["track_name"].fillna("Unknown Title").replace({"NA": "NA (Single)"})
df_master = df_master.reindex(columns=EXACT_COLUMNS)

df_master.to_csv(OUTPUT_CSV, index=False)

print("=" * 60)
print(f"SUCCESS! Created accurate {OUTPUT_CSV}")
print(f"Total rows: {len(df_master):,}")
print(f"Total null values: {df_master.isnull().sum().sum()}")
print("\nTop 15 Genre distribution:")
print(df_master["track_genre"].value_counts().head(15))
print("=" * 60)
